# ULIP-2 scoring (point-cloud ↔ text) on Colab

Scores each mesh's point cloud against its caption with **ULIP-2 colored PointBERT** — the 3D metric Twist & Compute reports.

**Runtime → GPU.** No CUDA extensions to build and no runtime restart: the only CUDA op ULIP needs (furthest-point-sampling) is swapped for a pure-torch version that runs on the GPU.

Flow: locally run `eval/export_pointclouds.py <results.csv>` → zip `eval/pointclouds/` → upload here → score → download `ulip_scores.csv` → locally `eval/merge_ulip.py <results.csv> ulip_scores.csv`.

In [ ]:
!nvidia-smi -L

In [ ]:
# 1. Clone ULIP, make the CUDA-only / unused imports optional, install python deps.
#    No pointnet2_ops, no open3d, no numpy downgrade -> no build, no restart.
#    Idempotent: each patch checks before writing so re-runs are safe.
%cd /content
!git clone --depth 1 https://github.com/salesforce/ULIP.git 2>/dev/null || echo "ULIP already cloned"
import pathlib

d = pathlib.Path('/content/ULIP/models/pointbert/dvae.py'); s = d.read_text()
if 'KNN=None' not in s:
    s = s.replace('from knn_cuda import KNN', 'try:\n    from knn_cuda import KNN\nexcept Exception:\n    KNN=None')
    s = s.replace('knn = KNN(k=4, transpose_mode=False)', 'knn = None')
    d.write_text(s)

m = pathlib.Path('/content/ULIP/models/pointbert/misc.py'); s = m.read_text()
if 'pointnet2_utils=None' not in s:
    s = s.replace('from pointnet2_ops import pointnet2_utils', 'try:\n    from pointnet2_ops import pointnet2_utils\nexcept Exception:\n    pointnet2_utils=None')
    m.write_text(s)

io = pathlib.Path('/content/ULIP/utils/io.py'); s = io.read_text()
if 'open3d=None' not in s:
    s = s.replace('import open3d', 'try:\n    import open3d\nexcept Exception:\n    open3d=None')
    io.write_text(s)

!pip install open_clip_torch==2.24.0 timm easydict pyyaml 'huggingface_hub[hf_transfer]'

In [ ]:
# 2. Patch FPS (pure-torch) + build/load the ULIP-2 colored PointBERT model.
import os, sys, types, torch

# torch._six was removed in PyTorch 1.9; ULIP still imports string_classes from it
if 'torch._six' not in sys.modules:
    _six = types.ModuleType('torch._six')
    _six.string_classes = (str,)
    _six.inf = float('inf')
    sys.modules['torch._six'] = _six

sys.path.insert(0, '/content/ULIP')
os.chdir('/content/ULIP')  # config yaml path is relative to here
from easydict import EasyDict
from huggingface_hub import hf_hub_download
from models.ULIP_models import ULIP2_PointBERT_Colored

import models.pointbert.misc as _misc
def _fps(data, number):
    """Pure-torch furthest point sampling (replaces pointnet2_ops). data: (B,N,C)."""
    B, N, C = data.shape
    xyz = data[:, :, :3]; dev = data.device
    centroids = torch.zeros(B, number, dtype=torch.long, device=dev)
    distance = torch.ones(B, N, device=dev) * 1e10
    farthest = torch.randint(0, N, (B,), dtype=torch.long, device=dev)
    batch = torch.arange(B, dtype=torch.long, device=dev)
    for i in range(number):
        centroids[:, i] = farthest
        c = xyz[batch, farthest, :].view(B, 1, 3)
        dist = ((xyz - c) ** 2).sum(-1)
        mask = dist < distance; distance[mask] = dist[mask]
        farthest = distance.max(-1)[1]
    return torch.gather(data, 1, centroids.unsqueeze(-1).expand(-1, -1, C))
_misc.fps = _fps
print('FPS patched (pure-torch, no pointnet2_ops)')

model = ULIP2_PointBERT_Colored(EasyDict(evaluate_3d=True, npoints=10000)).cuda().eval()
ckpt = hf_hub_download(
    repo_id='SFXX/ulip', repo_type='dataset',
    filename='ULIP-2/pretrained_models/ULIP-2-PointBERT-10k-xyzrgb-pc-vit_g-objaverse_shapenet-pretrained.pt')
sd = torch.load(ckpt, map_location='cpu', weights_only=False)  # checkpoint has numpy scalars
sd = sd.get('state_dict', sd)
sd = {k.replace('module.', ''): v for k, v in sd.items()}
rep = model.load_state_dict(sd, strict=False)
print(f'loaded; missing={len(rep.missing_keys)} unexpected={len(rep.unexpected_keys)}')

In [ ]:
# 3. Upload pointclouds.zip (locally: export_pointclouds.py then zip eval/pointclouds/).
from google.colab import files
import zipfile, glob, os, csv
os.makedirs('/content/pc', exist_ok=True)
for name in files.upload():
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('/content/pc')
man = glob.glob('/content/pc/**/manifest.csv', recursive=True)[0]
base = os.path.dirname(man)
rows = list(csv.DictReader(open(man)))
print(f'{len(rows)} point clouds to score')

In [ ]:
# 4. Score each point cloud vs its caption -> ulip_scores.csv
import numpy as np, csv, os
out = '/content/ulip_scores.csv'
with open(out, 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=['key', 'ulip']); w.writeheader()
    for i, r in enumerate(rows, 1):
        key, caption = r['key'], r['caption']
        try:
            pc = torch.as_tensor(np.load(os.path.join(base, key + '.npy')),
                                 dtype=torch.float32, device='cuda').unsqueeze(0)
            with torch.no_grad():
                pe = model.encode_pc(pc); pe = pe / pe.norm(dim=-1, keepdim=True)
                te = model.encode_text(model.tokenizer([caption]).cuda())
                te = te / te.norm(dim=-1, keepdim=True)
                sim = float((pe @ te.T).item())
            w.writerow({'key': key, 'ulip': f'{sim:.4f}'})
            print(f'[{i}/{len(rows)}] {key}: {sim:.4f}')
        except Exception as e:
            print(f'[{i}/{len(rows)}] {key}: FAILED {e}')
            w.writerow({'key': key, 'ulip': ''})
from google.colab import files
files.download(out)